In [2]:
from pykeen.triples import TriplesFactory
import pandas as pd
from rdflib import graph, Namespace, URIRef
from rdflib.namespace import RDF

import torch
import csv
from tqdm import tqdm # A library for a smart progress bar

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# YAGO 4.5-10

In [3]:
#first do sed 's/ \.$//' yago_triples.nt > yago_triples.txt

tf = TriplesFactory.from_path('YAGO4.5/yago_triples.txt', create_inverse_triples=False, load_triples_kwargs={'delimiter':' '})
training, testing, validation = tf.split([.999, .0005, .0005],random_state=42)

In [ ]:
# pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('YAGO4.5/YAGO4-5-10_train.txt', header=False, index=False, sep ='\t')
# pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('YAGO4.5/YAGO4-5-10_test.txt', header=False, index=False, sep ='\t')
# pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('YAGO4.5/YAGO4-5-10_valid.txt', header=False, index=False, sep ='\t')

In [ ]:
# df = pd.DataFrame(training.label_triples(training.mapped_triples))
# df.to_csv('YAGO4.5/YAGO4-5-10_train.txt', header=False, index=False, sep ='\t')

In [4]:
e_conversion_dict = {value: key for key, value in training.entity_to_id.items()}
r_conversion_dict = {value: key for key, value in training.relation_to_id.items()}

In [7]:
output_filename = 'YAGO4.5/YAGO4-5-10_train.txt'
input_triples= training.mapped_triples
batch_size = 100_000
with open(output_filename, 'w', newline='', encoding='utf-8') as f:
    # Create a CSV writer with a tab delimiter
    writer = csv.writer(f, delimiter='\t')

    # Use tqdm for a helpful progress bar
    # We iterate through the tensor in steps of batch_size
    for i in tqdm(range(0, input_triples.shape[0], batch_size)):
        # Get a chunk of the tensor
        chunk = input_triples[i : i + batch_size]

        # Move chunk to CPU (if it's on GPU) and convert to a Python list
        # .tolist() is efficient for converting a small chunk
        chunk_list = chunk.cpu().tolist()

        # Map the integer values to strings using the dictionary.
        # We use .get() for safety in case a key is missing.
        mapped_rows = [
            [e_conversion_dict.get(row[0], 'KEY_NOT_FOUND'),
             r_conversion_dict.get(row[1], 'KEY_NOT_FOUND'),
             e_conversion_dict.get(row[2], 'KEY_NOT_FOUND')]
            for row in chunk_list
        ]

        # Write the mapped rows to the TSV file
        writer.writerows(mapped_rows)

100%|██████████| 156/156 [00:47<00:00,  3.26it/s]


# NELL995 splits

In [ ]:
default_ns = 'https://ste-lod-crew.fr/nell/ontology/'
ns = Namespace(default_ns)
g = graph.Graph()

with open('NELL995/NELLKG0.txt') as inFile:
    with open('NELL995/NELLKG0_with_IRI.txt', 'w') as outFile:
        for line in inFile:
            line=  line.replace('__','_')
            s,p,o = line.split()
            sClass, sName = s.split('_',1)
            oClass, oName = o.split('_',1)

            sClass = default_ns+sClass
            sName = default_ns+sName
            oClass = default_ns+oClass
            oName = default_ns+oName
            pName = default_ns+p

            outFile.write(sClass+'_'+sName + '\t' + pName + '\t' + oClass+'_'+oName + '\n')
            g.add((URIRef(sName), RDF.type, URIRef(sClass)))
            g.add((URIRef(oName), RDF.type, URIRef(oClass)))
            g.add((URIRef(sName), URIRef(pName), URIRef(oName)))

    g.serialize('../datasets/NELL995/NELLKG0_with_IRI.ttl', format='turtle')



In [ ]:
tf = TriplesFactory.from_path('NELL995/NELLKG0_with_IRI.txt')
training, testing, validation = tf.split([.8, .1, .1],random_state=42)

In [35]:
pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('NELL995/NELL995_train.txt', header=False, index=False, sep ='\t')
pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('NELL995/NELL995_test.txt', header=False, index=False, sep ='\t')
pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('NELL995/NELL995_valid.txt', header=False, index=False, sep ='\t')
# pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_train.txt', header=False, index=False, sep = '\t')
# pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_test.txt', header=False, index=False, sep = '\t')
# pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_valid.txt', header=False, index=False, sep = '\t')
